# Stage 3: Structured Logit Calibration (RQ2)
**PsyDefDetect Research Pipeline — Notebook 04**

**Research Question (RQ2):** Does structured calibration with compact parameters narrow the generalization gap vs random bias search?

**Inputs:**
- `artifacts/splits/train_folds.parquet`
- `artifacts/stage1_qwen3_8b/seed_42/oof_logits.npy` — shape [1864, 9]
- `artifacts/stage2_sequential/oof_logits_lgbm.npy` — shape [1864, 9]
- `artifacts/stage2_sequential/oof_logits_catboost.npy` — shape [1864, 9]
- `artifacts/stage2_sequential/test_logits_routed_lgbm.npy` — shape [472, 9]
- `artifacts/stage2_sequential/test_logits_routed_catboost.npy` — shape [472, 9]
- `artifacts/stage1_qwen3_8b/seed_42/test_logits_by_fold.npy` — shape [5, 472, 9]

**Outputs:**
- `artifacts/results/calibration_nested_cv.csv`
- `artifacts/results/best_calibration_params.json`
- `artifacts/results/reliability_diagrams.png`


## Section 1 — Imports & Config


In [ ]:
import os
import json
import warnings
import itertools
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.special as ssp
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
np.random.seed(42)

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10

print('Imports OK')


In [ ]:
from pathlib import Path
import os
@dataclass
class CFG:
    SEEDS: List[int] = field(default_factory=lambda: [42, 2026, 777])
    MODEL_NAME_SMALL: str = 'Qwen/Qwen3-4B'
    MODEL_NAME_MAIN: str = 'Qwen/Qwen3-8B'
    DATA_DIR: str = "."
    ARTIFACT_DIR: str = "artifacts"
    N_FOLDS: int = 5
    MAX_HISTORY_TURNS: int = 30
    MAX_LENGTH: int = 1536
    N_CLASSES: int = 9
    EVAL_CLASSES: List[int] = field(default_factory=lambda: list(range(1, 9)))  # classes 1–8
    RANDOM_BIAS_TRIALS: int = 22000
    RANDOM_BIAS_RANGE: float = 2.0

cfg = CFG()

# Create output directories
for subdir in ['splits', 'stage1_qwen3_8b/seed_42', 'stage2_sequential', 'results']:
    os.makedirs(os.path.join(cfg.ARTIFACT_DIR, subdir), exist_ok=True)

print('CFG:', cfg)
print('Artifact dir:', cfg.ARTIFACT_DIR)


## Section 2 — Calibration Utility Functions


In [ ]:
# ---------------------------------------------------------------------------
# Core calibration interface
# ---------------------------------------------------------------------------

def compute_class_priors(labels: np.ndarray, n_classes: int = 9, eps: float = 1e-8) -> np.ndarray:
    """Compute smoothed class prior probabilities from integer label array."""
    counts = np.bincount(labels.astype(int), minlength=n_classes).astype(float)
    counts += eps  # Laplace-like smoothing to avoid log(0)
    return counts / counts.sum()


def calibrate_logits(
    logits: np.ndarray,
    params: Dict[str, Any],
    method: str,
    priors: Optional[np.ndarray] = None,
    lr_model: Optional[LogisticRegression] = None,
) -> np.ndarray:
    """
    Apply calibration to logits.

    Args:
        logits   : shape [N, 9]
        params   : method-specific parameter dict
        method   : one of 'A', 'B', 'C', 'D', 'random'
        priors   : class priors, shape [9]  (required for A, B, C, random)
        lr_model : fitted LogisticRegression (required for D)

    Returns:
        calibrated logits, shape [N, 9]
    """
    z = logits.copy().astype(float)

    if method == 'A':
        # z'_c = z_c - tau * log(pi_c)
        tau = params['tau']
        log_pi = np.log(priors + 1e-12)  # [9]
        z = z - tau * log_pi[np.newaxis, :]  # broadcast

    elif method == 'B':
        # z'_c = z_c / T - tau * log(pi_c)
        T = params['T']
        tau = params['tau']
        log_pi = np.log(priors + 1e-12)
        z = z / T - tau * log_pi[np.newaxis, :]

    elif method == 'C':
        # z'_0 = z_0 + b0
        # z'_7 = z_7 + b7
        # z'_c = z_c - tau * log(pi_c)  for c in {1..6, 8}
        b0 = params['b0']
        b7 = params['b7']
        tau = params['tau']
        log_pi = np.log(priors + 1e-12)
        z_new = z.copy()
        z_new[:, 0] = z[:, 0] + b0
        z_new[:, 7] = z[:, 7] + b7
        for c in list(range(1, 7)) + [8]:
            z_new[:, c] = z[:, c] - tau * log_pi[c]
        z = z_new

    elif method == 'D':
        # Platt scaling: return decision_function (log-odds) from fitted LR
        if lr_model is None:
            raise ValueError('lr_model required for method D')
        z = lr_model.decision_function(logits)

    elif method == 'random':
        # z'_c = z_c + delta_c
        delta = np.array(params['delta'])  # shape [9]
        z = z + delta[np.newaxis, :]

    else:
        raise ValueError(f'Unknown calibration method: {method}')

    return z


print('calibrate_logits defined')


In [ ]:
# ---------------------------------------------------------------------------
# Macro-F1 on classes 1–8  (the main evaluation metric throughout)
# ---------------------------------------------------------------------------

EVAL_CLASSES = list(range(1, 9))  # classes 1–8

def macro_f1_1_8(y_true: np.ndarray, logits_cal: np.ndarray) -> float:
    """
    Compute macro-averaged F1 on classes 1–8 from calibrated logits.
    If a class is entirely absent from y_true, it's skipped (zero_division=0).
    """
    y_pred = np.argmax(logits_cal, axis=1)
    # Restrict to classes present in y_true (but average over EVAL_CLASSES)
    return f1_score(
        y_true,
        y_pred,
        labels=EVAL_CLASSES,
        average='macro',
        zero_division=0,
    )


print('macro_f1_1_8 defined')


In [ ]:
# ---------------------------------------------------------------------------
# Method A inner CV tuning: grid search over tau
# ---------------------------------------------------------------------------

TAU_GRID_A = np.linspace(0.0, 2.0, 41)

def tune_method_A(
    train_logits: np.ndarray,
    train_labels: np.ndarray,
    val_logits: np.ndarray,
    val_labels: np.ndarray,
) -> Dict[str, float]:
    """Grid search tau for Method A. Returns best params dict."""
    priors = compute_class_priors(train_labels)
    best_tau, best_f1 = 0.0, -1.0
    for tau in TAU_GRID_A:
        cal = calibrate_logits(val_logits, {'tau': tau}, 'A', priors=priors)
        f1 = macro_f1_1_8(val_labels, cal)
        if f1 > best_f1:
            best_f1 = f1
            best_tau = tau
    return {'tau': best_tau}, best_f1


# ---------------------------------------------------------------------------
# Method B inner CV tuning
# ---------------------------------------------------------------------------

T_GRID_B   = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
TAU_GRID_B = np.linspace(0.0, 2.0, 21)

def tune_method_B(
    train_logits: np.ndarray,
    train_labels: np.ndarray,
    val_logits: np.ndarray,
    val_labels: np.ndarray,
) -> Tuple[Dict[str, float], float]:
    priors = compute_class_priors(train_labels)
    best_params, best_f1 = {'T': 1.0, 'tau': 0.0}, -1.0
    for T in T_GRID_B:
        for tau in TAU_GRID_B:
            cal = calibrate_logits(val_logits, {'T': T, 'tau': tau}, 'B', priors=priors)
            f1 = macro_f1_1_8(val_labels, cal)
            if f1 > best_f1:
                best_f1 = f1
                best_params = {'T': T, 'tau': tau}
    return best_params, best_f1


# ---------------------------------------------------------------------------
# Method C inner CV tuning
# ---------------------------------------------------------------------------

B0_GRID_C  = np.linspace(-1.5, 1.5, 11)
B7_GRID_C  = np.linspace(-1.5, 1.5, 11)
TAU_GRID_C = np.linspace(0.0, 2.0, 11)

def tune_method_C(
    train_logits: np.ndarray,
    train_labels: np.ndarray,
    val_logits: np.ndarray,
    val_labels: np.ndarray,
) -> Tuple[Dict[str, float], float]:
    priors = compute_class_priors(train_labels)
    log_pi = np.log(priors + 1e-12)
    best_params = {'b0': 0.0, 'b7': 0.0, 'tau': 0.0}
    best_f1 = -1.0

    # Pre-compute the prior-shifted part for classes 1..6, 8
    # z'_c = z_c - tau * log_pi[c] for those classes — depends on tau
    # We iterate over (b0, b7, tau)
    for b0, b7, tau in itertools.product(B0_GRID_C, B7_GRID_C, TAU_GRID_C):
        cal = calibrate_logits(
            val_logits, {'b0': b0, 'b7': b7, 'tau': tau}, 'C', priors=priors
        )
        f1 = macro_f1_1_8(val_labels, cal)
        if f1 > best_f1:
            best_f1 = f1
            best_params = {'b0': b0, 'b7': b7, 'tau': tau}
    return best_params, best_f1


# ---------------------------------------------------------------------------
# Method D inner CV: fit logistic regression, return model
# ---------------------------------------------------------------------------

def tune_method_D(
    train_logits: np.ndarray,
    train_labels: np.ndarray,
    val_logits: np.ndarray,
    val_labels: np.ndarray,
) -> Tuple[Dict[str, Any], float, LogisticRegression]:
    model = LogisticRegression(
        C=1.0, multi_class='multinomial', solver='lbfgs', max_iter=1000
    )
    # Handle edge-case: if some class absent in train, add a dummy row
    unique_classes = np.unique(train_labels)
    if len(unique_classes) < 9:
        missing = [c for c in range(9) if c not in unique_classes]
        dummy_logits = np.zeros((len(missing), 9))
        dummy_labels = np.array(missing)
        aug_logits = np.vstack([train_logits, dummy_logits])
        aug_labels = np.concatenate([train_labels, dummy_labels])
    else:
        aug_logits, aug_labels = train_logits, train_labels
    model.fit(aug_logits, aug_labels)
    cal = calibrate_logits(val_logits, {}, 'D', lr_model=model)
    f1 = macro_f1_1_8(val_labels, cal)
    params = {'coef': model.coef_.tolist(), 'intercept': model.intercept_.tolist()}
    return params, f1, model


# ---------------------------------------------------------------------------
# Random Bias Search
# ---------------------------------------------------------------------------

def tune_random_bias(
    train_logits: np.ndarray,
    train_labels: np.ndarray,
    val_logits: np.ndarray,
    val_labels: np.ndarray,
    n_trials: int = 22000,
    bias_range: float = 2.0,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[Dict[str, Any], float]:
    """Vectorized fast random bias search across 22,000 trials."""
    if rng is None:
        rng = np.random.default_rng(42)
    deltas = rng.uniform(-bias_range, bias_range, size=(n_trials, 9))
    
    target_classes = np.arange(1, 9)
    y_true_onehot = (val_labels[:, None] == target_classes[None, :])
    true_pos_weights = y_true_onehot.sum(axis=0)
    
    chunk_size = 2000
    best_f1 = -1.0
    best_delta = deltas[0]
    
    for i in range(0, len(deltas), chunk_size):
        chunk_deltas = deltas[i:i+chunk_size]
        chunk_logits = val_logits[None, :, :] + chunk_deltas[:, None, :]
        preds = np.argmax(chunk_logits, axis=-1)
        
        preds_onehot = (preds[:, :, None] == target_classes[None, None, :])
        tp = np.logical_and(preds_onehot, y_true_onehot[None, :, :]).sum(axis=1)
        pred_pos = preds_onehot.sum(axis=1)
        
        prec = np.divide(tp, pred_pos, out=np.zeros_like(tp, dtype=float), where=pred_pos > 0)
        rec = np.divide(tp, true_pos_weights[None, :], out=np.zeros_like(tp, dtype=float), where=true_pos_weights[None, :] > 0)
        denom = prec + rec
        f1_c = np.divide(2 * prec * rec, denom, out=np.zeros_like(prec, dtype=float), where=denom > 0)
        macro_f1 = f1_c.mean(axis=1)
        
        max_idx = np.argmax(macro_f1)
        if macro_f1[max_idx] > best_f1:
            best_f1 = float(macro_f1[max_idx])
            best_delta = chunk_deltas[max_idx].copy()
            
    return {'delta': best_delta.tolist()}, best_f1


print('All calibration tuning functions defined')


## Section 3 — Nested Grouped Cross-Validation Protocol (RQ2 core)


In [ ]:
def _inner_cv_best_params(
    method: str,
    logits: np.ndarray,
    labels: np.ndarray,
    dialogue_ids: np.ndarray,
    inner_n: int = 4,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[Any, float, Optional[Any]]:
    """
    Run inner CV for a single outer-train slice to find best calibration params.
    Returns (best_params, mean_inner_f1, best_lr_model|None).
    """
    if rng is None:
        rng = np.random.default_rng(42)

    sgkf_inner = StratifiedGroupKFold(n_splits=inner_n, shuffle=True, random_state=42)

    # Collect per-inner-fold best params and scores
    fold_params_list = []
    fold_f1_list = []
    fold_model_list = []  # only for method D

    for inner_tr_idx, inner_val_idx in sgkf_inner.split(logits, labels, groups=dialogue_ids):
        if len(inner_tr_idx) == 0 or len(inner_val_idx) == 0:
            continue
        tr_log = logits[inner_tr_idx]
        tr_lab = labels[inner_tr_idx]
        val_log = logits[inner_val_idx]
        val_lab = labels[inner_val_idx]

        if method == 'A':
            params, f1 = tune_method_A(tr_log, tr_lab, val_log, val_lab)
            fold_params_list.append(params)
            fold_f1_list.append(f1)

        elif method == 'B':
            params, f1 = tune_method_B(tr_log, tr_lab, val_log, val_lab)
            fold_params_list.append(params)
            fold_f1_list.append(f1)

        elif method == 'C':
            params, f1 = tune_method_C(tr_log, tr_lab, val_log, val_lab)
            fold_params_list.append(params)
            fold_f1_list.append(f1)

        elif method == 'D':
            params, f1, model = tune_method_D(tr_log, tr_lab, val_log, val_lab)
            fold_params_list.append(params)
            fold_f1_list.append(f1)
            fold_model_list.append(model)

        elif method == 'random':
            params, f1 = tune_random_bias(
                tr_log, tr_lab, val_log, val_lab,
                n_trials=cfg.RANDOM_BIAS_TRIALS,
                bias_range=cfg.RANDOM_BIAS_RANGE,
                rng=rng,
            )
            fold_params_list.append(params)
            fold_f1_list.append(f1)

        else:
            raise ValueError(f'Unknown method: {method}')

    if not fold_f1_list:
        return {}, 0.0, None

    mean_inner_f1 = float(np.mean(fold_f1_list))

    # Choose best params: use the inner fold with highest f1
    best_idx = int(np.argmax(fold_f1_list))
    best_params = fold_params_list[best_idx]
    best_model = fold_model_list[best_idx] if method == 'D' and fold_model_list else None

    # For scalar-param methods, use median across folds for robustness
    if method == 'A':
        best_params = {'tau': float(np.median([p['tau'] for p in fold_params_list]))}
    elif method == 'B':
        best_params = {
            'T': float(np.median([p['T'] for p in fold_params_list])),
            'tau': float(np.median([p['tau'] for p in fold_params_list])),
        }
    elif method == 'C':
        best_params = {
            'b0': float(np.median([p['b0'] for p in fold_params_list])),
            'b7': float(np.median([p['b7'] for p in fold_params_list])),
            'tau': float(np.median([p['tau'] for p in fold_params_list])),
        }
    elif method == 'random':
        # Median of delta vectors
        delta_arr = np.array([p['delta'] for p in fold_params_list])  # [n_inner, 9]
        best_params = {'delta': np.median(delta_arr, axis=0).tolist()}

    return best_params, mean_inner_f1, best_model


def nested_grouped_cv(
    logits: np.ndarray,
    labels: np.ndarray,
    dialogue_ids: np.ndarray,
    fold_df: pd.DataFrame,
    outer_n: int = 5,
    inner_n: int = 4,
    methods: Optional[List[str]] = None,
    rng_seed: int = 42,
) -> pd.DataFrame:
    """
    Nested grouped cross-validation for calibration method comparison (RQ2).

    Args:
        logits       : [N, 9] OOF logits
        labels       : [N]    integer labels
        dialogue_ids : [N]    dialogue ID strings
        fold_df      : DataFrame with columns [id, dialogue_id, label, fold_id, ...]
                       Same ordering as logits (row i <-> fold_df.iloc[i]).
        outer_n      : number of outer folds (matches Stage 1 fold_id values 0..4)
        inner_n      : number of inner folds
        methods      : list of method names; default = ['A', 'B', 'C', 'D', 'random']

    Returns:
        DataFrame with columns: method, outer_fold, inner_mean_f1, outer_f1, gap
    """
    if methods is None:
        methods = ['A', 'B', 'C', 'D', 'random']

    rng = np.random.default_rng(rng_seed)
    records = []

    fold_ids = fold_df['fold_id'].values  # [N], same ordering
    unique_outer_folds = sorted(fold_df['fold_id'].unique())

    for outer_fold in unique_outer_folds:
        outer_val_mask = (fold_ids == outer_fold)
        outer_tr_mask  = ~outer_val_mask

        outer_tr_logits  = logits[outer_tr_mask]
        outer_tr_labels  = labels[outer_tr_mask]
        outer_tr_dlg_ids = dialogue_ids[outer_tr_mask]

        outer_val_logits = logits[outer_val_mask]
        outer_val_labels = labels[outer_val_mask]

        print(f'  Outer fold {outer_fold}: train={outer_tr_mask.sum()}, val={outer_val_mask.sum()}')

        for method in methods:
            print(f'    Method {method} ...', end=' ', flush=True)

            # --- Inner CV to find best params ---
            best_params, mean_inner_f1, best_model = _inner_cv_best_params(
                method=method,
                logits=outer_tr_logits,
                labels=outer_tr_labels,
                dialogue_ids=outer_tr_dlg_ids,
                inner_n=inner_n,
                rng=rng,
            )

            # --- Apply best params to outer val fold ---
            if method == 'D':
                # Refit on full outer-train with best LR (C=1.0 fixed)
                _, _, best_model = tune_method_D(
                    outer_tr_logits, outer_tr_labels,
                    outer_val_logits, outer_val_labels,
                )
                cal_val = calibrate_logits(outer_val_logits, {}, 'D', lr_model=best_model)
            else:
                priors_outer = compute_class_priors(outer_tr_labels)
                cal_val = calibrate_logits(
                    outer_val_logits, best_params, method, priors=priors_outer
                )

            outer_f1 = macro_f1_1_8(outer_val_labels, cal_val)
            gap = mean_inner_f1 - outer_f1

            print(f'inner={mean_inner_f1:.4f} outer={outer_f1:.4f} gap={gap:+.4f}')

            records.append({
                'method': method,
                'outer_fold': outer_fold,
                'inner_mean_f1': mean_inner_f1,
                'outer_f1': outer_f1,
                'gap': gap,
                'best_params': json.dumps(best_params),
            })

    return pd.DataFrame(records)


print('nested_grouped_cv defined')


## Section 4 — Load Data & Run Nested CV


In [ ]:
# ---------------------------------------------------------------------------
# Load all artifacts
# ---------------------------------------------------------------------------

ART = cfg.ARTIFACT_DIR

# Fold DataFrame
fold_df = pd.read_parquet(os.path.join(ART, 'splits', 'train_folds.parquet'))
print('fold_df shape:', fold_df.shape)
print(fold_df.head(3))

labels       = fold_df['label'].values.astype(int)
dialogue_ids = fold_df['dialogue_id'].values

# Stage 1 OOF logits
s1_oof_logits = np.load(os.path.join(ART, 'stage1_qwen3_8b', 'seed_42', 'oof_logits.npy'))
print(f'Stage 1 OOF logits: {s1_oof_logits.shape}')

# Stage 2 OOF logits
s2_lgbm_oof   = np.load(os.path.join(ART, 'stage2_sequential', 'oof_logits_lgbm.npy'))
s2_cat_oof    = np.load(os.path.join(ART, 'stage2_sequential', 'oof_logits_catboost.npy'))
print(f'Stage 2 LGBM OOF:  {s2_lgbm_oof.shape}')
print(f'Stage 2 Cat  OOF:  {s2_cat_oof.shape}')

# Stage 2 test logits
s2_lgbm_test  = np.load(os.path.join(ART, 'stage2_sequential', 'test_logits_routed_lgbm.npy'))
s2_cat_test   = np.load(os.path.join(ART, 'stage2_sequential', 'test_logits_routed_catboost.npy'))
s1_test_by_fold = np.load(os.path.join(ART, 'stage1_qwen3_8b', 'seed_42', 'test_logits_by_fold.npy'))
print(f'Stage 2 LGBM test: {s2_lgbm_test.shape}')
print(f'Stage 2 Cat  test: {s2_cat_test.shape}')
print(f'Stage 1 test by fold: {s1_test_by_fold.shape}')

assert s1_oof_logits.shape == (1864, 9), f'Expected (1864,9) got {s1_oof_logits.shape}'
assert s2_lgbm_oof.shape  == (1864, 9), f'Expected (1864,9) got {s2_lgbm_oof.shape}'
assert s2_cat_oof.shape   == (1864, 9)
print('\nAll assertions passed.')


In [ ]:
# ---------------------------------------------------------------------------
# Run nested CV on three logit sources
# ---------------------------------------------------------------------------

METHODS = ['A', 'B', 'C', 'D', 'random']

logit_sources = {
    'stage1_oof':   s1_oof_logits,
    'stage2_lgbm':  s2_lgbm_oof,
    'stage2_catboost': s2_cat_oof,
}

all_results = []

for source_name, logits in logit_sources.items():
    print(f'\n========== Logit source: {source_name} ==========')
    df_cv = nested_grouped_cv(
        logits=logits,
        labels=labels,
        dialogue_ids=dialogue_ids,
        fold_df=fold_df,
        outer_n=5,
        inner_n=4,
        methods=METHODS,
        rng_seed=42,
    )
    df_cv.insert(0, 'logit_source', source_name)
    all_results.append(df_cv)
    print(f'  → Done. Shape: {df_cv.shape}')

results_df = pd.concat(all_results, ignore_index=True)
print('\nTotal result rows:', len(results_df))
results_df.head(10)


In [ ]:
# Save nested CV results
out_csv = os.path.join(ART, 'results', 'calibration_nested_cv.csv')
results_df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')


## Section 5 — RQ2 Summary Table


In [ ]:
# ---------------------------------------------------------------------------
# Summary: mean ± std per (logit_source × method) across 5 outer folds
# ---------------------------------------------------------------------------

summary = (
    results_df
    .groupby(['logit_source', 'method'])
    .agg(
        inner_f1_mean=('inner_mean_f1', 'mean'),
        inner_f1_std=('inner_mean_f1', 'std'),
        outer_f1_mean=('outer_f1', 'mean'),
        outer_f1_std=('outer_f1', 'std'),
        gap_mean=('gap', 'mean'),
        gap_std=('gap', 'std'),
    )
    .reset_index()
)

summary['inner_f1_str'] = summary.apply(
    lambda r: f"{r['inner_f1_mean']:.4f} ± {r['inner_f1_std']:.4f}", axis=1
)
summary['outer_f1_str'] = summary.apply(
    lambda r: f"{r['outer_f1_mean']:.4f} ± {r['outer_f1_std']:.4f}", axis=1
)
summary['gap_str'] = summary.apply(
    lambda r: f"{r['gap_mean']:+.4f} ± {r['gap_std']:.4f}", axis=1
)

display_cols = ['logit_source', 'method', 'inner_f1_str', 'outer_f1_str', 'gap_str']
print('\n=== RQ2 Summary Table (mean ± std across 5 outer folds) ===')
print(summary[display_cols].to_string(index=False))

# Highlight method with smallest |gap| per logit source
print('\n=== Method with smallest generalization gap per logit source ===')
best_gap = summary.loc[summary.groupby('logit_source')['gap_mean'].idxmin()]
print(best_gap[['logit_source', 'method', 'gap_mean', 'outer_f1_mean']].to_string(index=False))


In [ ]:
# Visualise summary: outer F1 and gap side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: outer F1
ax = axes[0]
for src in summary['logit_source'].unique():
    sub = summary[summary['logit_source'] == src]
    ax.bar(
        [f"{src}\n{m}" for m in sub['method']],
        sub['outer_f1_mean'],
        yerr=sub['outer_f1_std'],
        label=src,
        alpha=0.75,
        capsize=4,
    )
ax.set_title('Outer CV Macro-F1 (classes 1-8)')
ax.set_ylabel('Macro-F1')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=8)

# Panel 2: gap
ax = axes[1]
for src in summary['logit_source'].unique():
    sub = summary[summary['logit_source'] == src]
    ax.bar(
        [f"{src}\n{m}" for m in sub['method']],
        sub['gap_mean'],
        yerr=sub['gap_std'],
        label=src,
        alpha=0.75,
        capsize=4,
    )
ax.axhline(0, color='k', linestyle='--', linewidth=0.8)
ax.set_title('Generalisation Gap (inner - outer F1)')
ax.set_ylabel('Gap')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(ART, 'results', 'calibration_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved calibration_comparison.png')


## Section 6 — Best Calibration Parameters (for Notebook 05)


In [ ]:
# ---------------------------------------------------------------------------
# Refit each method on all 1864 samples using Stage 2 LGBM OOF logits.
# Use median of best-params found across outer folds.
# ---------------------------------------------------------------------------

lgbm_results = results_df[results_df['logit_source'] == 'stage2_lgbm'].copy()

def _extract_best_params_across_folds(lgbm_results_df: pd.DataFrame, method: str) -> Dict:
    """Parse best_params JSON from each outer fold and compute median."""
    subset = lgbm_results_df[lgbm_results_df['method'] == method].copy()
    parsed = [json.loads(p) for p in subset['best_params']]
    return parsed


# Full dataset priors
all_labels = labels  # 1864 samples
all_logits = s2_lgbm_oof
all_priors = compute_class_priors(all_labels)
print('Class priors (all 1864):', np.round(all_priors, 4))

best_calibration_params = {}

# ---- Method A ----
params_a_folds = _extract_best_params_across_folds(lgbm_results, 'A')
tau_a = float(np.median([p['tau'] for p in params_a_folds]))
best_calibration_params['method_a'] = {'tau': tau_a}
print(f'Method A: tau = {tau_a:.4f}')

# ---- Method B ----
params_b_folds = _extract_best_params_across_folds(lgbm_results, 'B')
T_b   = float(np.median([p['T']   for p in params_b_folds]))
tau_b = float(np.median([p['tau'] for p in params_b_folds]))
best_calibration_params['method_b'] = {'T': T_b, 'tau': tau_b}
print(f'Method B: T = {T_b:.4f}, tau = {tau_b:.4f}')

# ---- Method C ----
params_c_folds = _extract_best_params_across_folds(lgbm_results, 'C')
b0_c  = float(np.median([p['b0']  for p in params_c_folds]))
b7_c  = float(np.median([p['b7']  for p in params_c_folds]))
tau_c = float(np.median([p['tau'] for p in params_c_folds]))
best_calibration_params['method_c'] = {'b0': b0_c, 'b7': b7_c, 'tau': tau_c}
print(f'Method C: b0 = {b0_c:.4f}, b7 = {b7_c:.4f}, tau = {tau_c:.4f}')

# ---- Method D ----
lr_full = LogisticRegression(C=1.0, multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_full.fit(all_logits, all_labels)
best_calibration_params['method_d'] = {
    'coef': lr_full.coef_.tolist(),
    'intercept': lr_full.intercept_.tolist(),
}
print(f'Method D: LR coef shape = {lr_full.coef_.shape}')

# ---- Random Bias ----
params_r_folds = _extract_best_params_across_folds(lgbm_results, 'random')
delta_arr = np.array([p['delta'] for p in params_r_folds])  # [5, 9]
best_delta = np.median(delta_arr, axis=0).tolist()
best_calibration_params['random_bias'] = {'delta': best_delta}
print(f'Random bias: delta = {[round(d,4) for d in best_delta]}')

# Pick best overall structured calibration method (Method C is the primary structured gate)
best_calibration_params['best_method'] = 'C'
best_calibration_params['method'] = 'C'
best_calibration_params['params'] = best_calibration_params['method_c']

# ---- Save ----
out_json = os.path.join(ART, 'results', 'best_calibration_params.json')
with open(out_json, 'w') as f:
    json.dump(best_calibration_params, f, indent=2)
print(f'\nSaved: {out_json}')


In [ ]:
# Verify saved JSON
with open(out_json) as f:
    loaded = json.load(f)
print('Keys:', list(loaded.keys()))
for k, v in loaded.items():
    if k == 'method_d':
        print(f'  {k}: coef shape={len(v["coef"])}x{len(v["coef"][0])}, intercept len={len(v["intercept"])}')
    else:
        print(f'  {k}: {v}')


## Section 7 — Calibration Curves & Reliability Diagrams


In [ ]:
# ---------------------------------------------------------------------------
# Reliability diagrams: Stage 2 LGBM OOF before/after Method C calibration
# 10 bins, one sub-plot per class (9 classes), two curves each
# ---------------------------------------------------------------------------

N_BINS = 10
CLASS_NAMES = [
    'No Defense',           # 0
    'Action',               # 1
    'Maj Image-Dist',       # 2
    'Disavowal',            # 3
    'Min Image-Dist',       # 4
    'Neurotic',             # 5
    'Obsessional',          # 6
    'High-Adaptive',        # 7
    'Needs More Info',      # 8
]

# Apply Method C with best params to all 1864 OOF logits
lgbm_cal_c = calibrate_logits(
    s2_lgbm_oof,
    {'b0': b0_c, 'b7': b7_c, 'tau': tau_c},
    'C',
    priors=all_priors,
)

# Convert to probabilities via softmax
def softmax(z: np.ndarray) -> np.ndarray:
    z_shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

probs_before = softmax(s2_lgbm_oof)   # [1864, 9]
probs_after  = softmax(lgbm_cal_c)    # [1864, 9]

print(f'probs_before shape: {probs_before.shape}')
print(f'probs_after  shape: {probs_after.shape}')

# Binary labels per class (one-vs-rest)
y_onehot = np.eye(9, dtype=int)[all_labels]  # [1864, 9]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes_flat = axes.flatten()

for c in range(9):
    ax = axes_flat[c]
    y_bin = y_onehot[:, c]  # binary: is class c?

    # Before calibration
    try:
        frac_pos_bef, mean_pred_bef = calibration_curve(
            y_bin, probs_before[:, c], n_bins=N_BINS, strategy='uniform'
        )
        ax.plot(mean_pred_bef, frac_pos_bef, 's-', color='royalblue',
                label='Before (LGBM)', linewidth=1.5, markersize=4)
    except Exception as e:
        ax.text(0.5, 0.5, f'Before: {e}', transform=ax.transAxes, fontsize=7, ha='center')

    # After calibration (Method C)
    try:
        frac_pos_aft, mean_pred_aft = calibration_curve(
            y_bin, probs_after[:, c], n_bins=N_BINS, strategy='uniform'
        )
        ax.plot(mean_pred_aft, frac_pos_aft, 'D-', color='tomato',
                label='After Method C', linewidth=1.5, markersize=4)
    except Exception as e:
        ax.text(0.5, 0.6, f'After: {e}', transform=ax.transAxes, fontsize=7, ha='center')

    # Perfect calibration diagonal
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Perfect')

    # Class prior line
    ax.axhline(all_priors[c], color='gray', linestyle=':', linewidth=0.8,
               label=f'Prior={all_priors[c]:.3f}')

    ax.set_title(f'Class {c}: {CLASS_NAMES[c]}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Mean Predicted Prob.', fontsize=8)
    ax.set_ylabel('Fraction of Positives', fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Reliability Diagrams: Stage 2 LGBM OOF Before/After Method C Calibration\n'
    f'(b0={b0_c:.2f}, b7={b7_c:.2f}, τ={tau_c:.2f})  10 bins',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()

out_png = os.path.join(ART, 'results', 'reliability_diagrams.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')


## Section 8 — Final Artifact Verification


In [ ]:
# ---------------------------------------------------------------------------
# Verify all artifacts were written
# ---------------------------------------------------------------------------

expected_artifacts = [
    os.path.join(ART, 'results', 'calibration_nested_cv.csv'),
    os.path.join(ART, 'results', 'best_calibration_params.json'),
    os.path.join(ART, 'results', 'reliability_diagrams.png'),
    os.path.join(ART, 'results', 'calibration_comparison.png'),
]

print('Artifact verification:')
all_ok = True
for path in expected_artifacts:
    exists = os.path.isfile(path)
    size   = os.path.getsize(path) if exists else 0
    status = '✓' if exists else '✗'
    print(f'  {status} {os.path.basename(path)} ({size:,} bytes)')
    if not exists:
        all_ok = False

print('\nAll artifacts present:', all_ok)

# Quick sanity: load nested CV CSV and print shape
cv_df = pd.read_csv(os.path.join(ART, 'results', 'calibration_nested_cv.csv'))
print(f'\ncalibration_nested_cv.csv: {cv_df.shape}')
print(f'  logit_source values: {cv_df["logit_source"].unique().tolist()}')
print(f'  method values:       {cv_df["method"].unique().tolist()}')
print(f'  outer_fold values:   {sorted(cv_df["outer_fold"].unique().tolist())}')

# Quick sanity: load best params JSON
with open(os.path.join(ART, 'results', 'best_calibration_params.json')) as f:
    bp = json.load(f)
print(f'\nbest_calibration_params.json keys: {list(bp.keys())}')

print('\n=== Stage 3 Structured Calibration COMPLETE ===')


---
## RQ2 Answer Summary

The nested grouped cross-validation protocol tests whether structured calibration methods (A, B, C, D) with compact parameter sets narrow the generalization gap compared to the unconstrained random bias search (22,000 trials over 9 free parameters).

**Key findings** (populated after execution):
- Methods A/B/C/D have 1–3 scalar parameters, strongly regularizing the search space.
- Random bias search with 9 free parameters is prone to overfitting the inner CV fold.
- Method C (Structured C0/C7 Gate + Prior Adjustment) is designed to address the dominant class imbalance (class 7 ≈ 52% of data, class 0 ≈ 16%).
- Method D (Platt/LR) uses all logit dimensions but is regularized via L2.

See `calibration_comparison.png` for the full visual comparison and `calibration_nested_cv.csv` for per-fold details.
